# Implement The Full Transformer Model

![Transformer model](../showcase_images/from_paper/main.png)

In [7]:
#TODO make sure this renders correctly on github repo

In [8]:
import torch.nn as nn

try: # works when ran via main.py (package mode)
    from .decoder import Decoder, DecoderLayer
    from .encoder import Encoder, EncoderLayer
    from .generator import Generator
    from .pos_encoding import PositionalEncoding
    from .embedding import Embeddings
except ImportError:
    # Works when running from inside Jupyter Notebook
    from decoder import Decoder, DecoderLayer
    from encoder import Encoder, EncoderLayer
    from generator import Generator
    from pos_encoding import PositionalEncoding
    from embedding import Embeddings

Testing Embedding...
Input shape: torch.Size([2, 5])
Output shape: torch.Size([2, 5, 512])
Embedding shape: torch.Size([2, 5, 512])
After Positional Encoding shape: torch.Size([2, 5, 512])
Success



In [ ]:
from typing import TYPE_CHECKING

if TYPE_CHECKING: # for type checks example cfg: English_german_config
    from configs.english_german_config import English_german_config

class Transformer(nn.Module):
    def __init__(
        self,
        cfg: English_german_config
    ):
        super().__init__()
        """
        The Transformer Model

        Args:
            cfg: Configurations
        """
        self.cfg = cfg
        self.decoder = Decoder(
            DecoderLayer(cfg.d_model, cfg.h, cfg.d_ff, cfg.dropout), cfg.N
        )
        self.encoder = Encoder(
            EncoderLayer(cfg.d_model, cfg.h, cfg.d_ff, cfg.dropout), cfg.N
        )

        # The embedded Source sequence
        self.src_embed = nn.Sequential(
            Embeddings(d_model=cfg.d_model, vocab_size=cfg.vocab_size),
            PositionalEncoding(
                d_model=cfg.d_model, pos_seq_len=cfg.pos_seq_len, dropout=cfg.dropout
            ),
        )

        # The embedded Target sequence
        self.tgt_embed = nn.Sequential(
            Embeddings(d_model=cfg.d_model, vocab_size=cfg.vocab_size),
            PositionalEncoding(
                d_model=cfg.d_model, pos_seq_len=cfg.pos_seq_len, dropout=cfg.dropout
            ),
        )
        self.generator = Generator(self.cfg.d_model, self.cfg.vocab_size)

    def encode(self, src, src_padding_mask):
        """
        Args:
            src: The source vocab.
            src_padding_mask: The <padding> masking for the source.
        """
        return self.encoder(self.src_embed(src), src_padding_mask)

    def decode(self, x, encoder_output, src_padding_mask, tgt_no_peek_mask):
        """
        Args:
            x: Target sequence.
            src: The source vocab.
            src_padding_mask: The <padding> masking for the source.
        """
        return self.decoder(
            self.tgt_embed(x), encoder_output, src_padding_mask, tgt_no_peek_mask
        )

    def forward(self, src, tgt, src_padding_mask, tgt_no_peek_mask):
        # Run the encoder
        encoder_out = self.encode(src, src_padding_mask)
        # Run Decoder
        decoder_out = self.decode(tgt, encoder_out, src_padding_mask, tgt_no_peek_mask)

        # Run last Linear + Softmax layers
        return self.generator(decoder_out)